# MODULE-1,2

# This cell imports all the necessary libraries for web scraping, data manipulation, and file system operations: requests for making HTTP requests, BeautifulSoup for parsing HTML, pandas for data handling, time for adding delays, and os for interacting with the operating system.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

# This cell initializes key variables for web scraping. BASE_URL is the main page for country military rankings. OTHER_SOURCES is a dictionary where each key is a URL for a specific military metric (like 'Total Population' or 'Defense Budget'), and its corresponding value is the name you want to use for that data column. Finally, os.makedirs('html_debug', exist_ok=True) creates a folder named 'html_debug' if it doesn't already exist, which can be used to save HTML files for debugging scraped content.

In [2]:
BASE_URL = "https://www.globalfirepower.com/countries-listing.php"

OTHER_SOURCES = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'Total Population',
    'https://www.globalfirepower.com/available-military-manpower.php': 'Total Military Personnel',
    'https://www.globalfirepower.com/active-military-manpower.php': 'Active Personnel',
    'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'Reserve Personnel',
    'https://www.globalfirepower.com/aircraft-total.php': 'Total Aircraft Strength',
    'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'Total Helicopter Strength',
    'https://www.globalfirepower.com/armor-tanks-total.php': 'Total Tank Strength',
    'https://www.globalfirepower.com/navy-ships.php': 'Total Naval Assets',
    'https://www.globalfirepower.com/defense-spending-budget.php': 'Defense Budget'
}
# folder for debugging HTML (optional but recommended)
os.makedirs("html_debug", exist_ok=True)

# The get_soup function is a utility designed for web scraping. It takes a URL as input and performs the following actions:

It uses the requests library to send an HTTP GET request to the provided URL, simulating a web browser by setting a 'User-Agent' header. It also sets a timeout of 20 seconds to prevent indefinite waiting.
response.raise_for_status() checks if the request was successful. If there was an HTTP error (e.g., 404 Not Found, 500 Server Error), it will raise an exception.
Finally, it parses the HTML content of the response (response.text) using BeautifulSoup with the 'html.parser' to create a navigable parse tree. This BeautifulSoup object, commonly referred to as 'soup', allows for easy extraction of data from the HTML structure.

In [3]:
def get_soup(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

# This cell calls the get_soup function with the BASE_URL, which is https://www.globalfirepower.com/countries-listing.php. The get_soup function then fetches the HTML content from this URL, parses it using BeautifulSoup, and the result is displayed in the output. This action effectively retrieves the main page content for inspection and subsequent scraping.

In [4]:
get_soup(BASE_URL)


<!DOCTYPE html>

<html lang="en-US">
<head>
<meta charset="utf-8"/>
<title>2026 Military Strength Ranking</title>
<link href="https://www.globalfirepower.com/countries-listing.php" rel="canonical"/>
<link href="https://www.globalfirepower.com/imgs/design/logo.png" rel="image_src"/>
<style type="text/css">@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/cyrillic/100/normal.woff2);unicode-range:U+0301,U+0400-045F,U+0490-0491,U+04B0-04B1,U+2116;font-display:swap;}@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/greek/100/normal.woff2);unicode-range:U+0370-03FF;font-display:swap;}@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/vietnamese/100/normal.woff2);unicode-range:U+0102-0103,U+0110-0111,U+0128-0129,U+0168-0169,U+01A0-01A1,U+01AF-01B0,U+0300-0301,U+0303-0304,U+0308-0309,U+0323,U+0329,U+1EA0-1EF

# The get_country_list function is designed to scrape the main ranking page (BASE_URL) to extract each country's name, their overall rank, and their 'PowerIndex'. It does this by:

Calling get_soup(BASE_URL) to get the parsed HTML of the main page.
Selecting all div elements with the class recordsetContainer, as each of these contains data for a single country.
Iterating through each record found:
It extracts the country name from a span within div.longFormName.
It extracts the rank from a span within div.rankNumContainer.
It extracts the PowerIndex value from a span within div.pwrIndxContainer, removing the 'PwrIndx:' prefix.
It appends this data to all_country_data as a dictionary, ensuring country, rank, and pwrindx are all present before adding.
Finally, it returns a pandas.DataFrame created from all_country_data.
After defining the function, the cell calls country_df = get_country_list() to execute the scraping and store the results in the country_df DataFrame. print(country_df.head()) then displays the first few rows of this DataFrame.

In [5]:
def get_country_list():
    soup = get_soup(BASE_URL)

    records = soup.select("div.recordsetContainer")
    all_country_data = []

    for record in records:
        country_span = record.select_one("div.longFormName > span")
        rank_container = record.select_one("div.rankNumContainer > span")
        pwrindx_container = record.select_one("div.pwrIndxContainer > span")

        country = country_span.text.strip() if country_span else None
        rank = rank_container.text.strip() if rank_container else None
        pwrindx = pwrindx_container.text.replace('PwrIndx:', '').strip() if pwrindx_container else None

        if country and rank and pwrindx:
            all_country_data.append({"Country": country, "Rank": rank, "PowerIndex": pwrindx})

    return pd.DataFrame(all_country_data)

country_df = get_country_list()
print(country_df.head())

         Country Rank PowerIndex
0  United States    1     0.0741
1         Russia    2     0.0791
2          China    3     0.0919
3          India    4     0.1346
4    South Korea    5     0.1642


# This cell defines an older version of the get_country_list function. It retrieves the HTML content from the BASE_URL using get_soup, then selects all span elements within div.longFormName to extract only the long-form country names. These names are then collected into a list and finally returned as a Pandas DataFrame with a single column named 'Country'. This version did not extract rank or PowerIndex, which were added in a later iteration.

In [6]:
# def get_country_list():
#     soup = get_soup(BASE_URL)

#     # This targets ONLY the long-form country names you showed in the HTML
#     country_spans = soup.select("div.longFormName > span")

#     countries = []
#     for span in country_spans:
#         country = span.text.strip()
#         if country:
#             countries.append(country)

#     return pd.DataFrame({"Country": countries})

# This cell calls the get_country_list function (the older version that only retrieved country names) and stores the result in the country_df DataFrame.

In [7]:
country_df=get_country_list()

# This cell displays the country_df DataFrame, showing the list of countries that were scraped by the previous get_country_list function.

In [8]:
country_df

,Country,Rank,PowerIndex
0,United States,1,0.0741
1,Russia,2,0.0791
2,China,3,0.0919
3,India,4,0.1346
4,South Korea,5,0.1642
...,...,...,...
140,Liberia,141,3.9275
141,Suriname,142,4.0538
142,Central African Republic,143,4.2381
143,Beliz,144,4.3602


# This cell calls the get_soup function again with the BASE_URL, similar to a previous cell. It serves as a re-demonstration or a check for the main page's content.

In [9]:
get_soup('https://www.globalfirepower.com/countries-listing.php')


<!DOCTYPE html>

<html lang="en-US">
<head>
<meta charset="utf-8"/>
<title>2026 Military Strength Ranking</title>
<link href="https://www.globalfirepower.com/countries-listing.php" rel="canonical"/>
<link href="https://www.globalfirepower.com/imgs/design/logo.png" rel="image_src"/>
<style type="text/css">@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/latin-ext/100/normal.woff2);unicode-range:U+0100-02AF,U+0304,U+0308,U+0329,U+1E00-1E9F,U+1EF2-1EFF,U+2020,U+20A0-20AB,U+20AD-20CF,U+2113,U+2C60-2C7F,U+A720-A7FF;font-display:swap;}@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/latin/100/normal.woff2);unicode-range:U+0000-00FF,U+0131,U+0152-0153,U+02BB-02BC,U+02C6,U+02DA,U+02DC,U+0304,U+0308,U+0329,U+2000-206F,U+2074,U+20AC,U+2122,U+2191,U+2193,U+2212,U+2215,U+FEFF,U+FFFD;font-display:swap;}@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:10

# This cell demonstrates the get_soup function by fetching and printing the BeautifulSoup object for the 'available-military-manpower.php' page, allowing inspection of its HTML structure.

In [10]:
get_soup('https://www.globalfirepower.com/available-military-manpower.php')


<!DOCTYPE html>

<html lang="en-US">
<head>
<meta charset="utf-8"/>
<title>Total Available Manpower by Country (2026)</title>
<link href="https://www.globalfirepower.com/available-military-manpower.php" rel="canonical"/>
<link href="https://www.globalfirepower.com/imgs/design/logo.png" rel="image_src"/>
<style type="text/css">@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/vietnamese/100/normal.woff2);unicode-range:U+0102-0103,U+0110-0111,U+0128-0129,U+0168-0169,U+01A0-01A1,U+01AF-01B0,U+0300-0301,U+0303-0304,U+0308-0309,U+0323,U+0329,U+1EA0-1EF9,U+20AB;font-display:swap;}@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/greek/100/normal.woff2);unicode-range:U+0370-03FF;font-display:swap;}@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/latin/100/normal.woff2);unicode-range:U+0000-00FF,U+0131,U+0

# The scrape_metric_page function is designed to extract specific metric data (like population, military personnel, etc.) from individual country metric pages. It takes a URL, uses get_soup to parse the page, and then selects specific elements (country_span and value_span) to build a dictionary of country-to-metric values.

In [11]:
def scrape_metric_page(url):
    soup = get_soup(url)

    records = soup.select("div.recordsetContainer")
    metric_data = {}

    for record in records:
        # Country name (long form only)
        country_span = record.select_one("div.longFormName > span")

        # Metric value (inside valueContainer)
        value_span = record.select_one("div.valueContainer span span")

        if country_span and value_span:
            country = country_span.text.strip()
            value = value_span.text.strip()
            metric_data[country] = value

    return metric_data

# This cell uses the scrape_metric_page function to extract the 'available military manpower' data and then prints the resulting dictionary, showing country names mapped to their respective manpower figures.bold text

In [12]:
manpower = scrape_metric_page('https://www.globalfirepower.com/available-military-manpower.php')
print(manpower)

{'China': '764,123,366', 'India': '662,290,299', 'United States': '150,463,900', 'Indonesia': '137,965,608', 'Nigeria': '125,475,979', 'Brazil': '112,226,271', 'Pakistan': '108,516,336', 'Bangladesh': '82,661,620', 'Russia': '69,002,197', 'Mexico': '61,447,766', 'Ethiopia': '56,904,143', 'Vietnam': '54,994,667', 'Japan': '52,976,836', 'Philippines': '50,859,137', 'Iran': '49,496,685', 'Egypt': '45,611,372', 'Turkiye': '42,985,080', 'Democratic Republic of the Congo': '39,237,029', 'Germany': '38,694,786', 'Thailand': '36,358,919', 'United Kingdom': '31,491,165', 'Myanmar': '30,489,384', 'France': '30,084,820', 'South Africa': '27,803,618', 'Italy': '27,434,219', 'Sudan': '26,747,657', 'South Korea': '26,040,900', 'Colombia': '24,298,295', 'Algeria': '22,570,787', 'Spain': '21,748,999', 'Kenya': '21,551,160', 'Argentina': '20,677,529', 'Saudi Arabia': '19,003,104', 'Uzbekistan': '18,990,708', 'Poland': '18,985,692', 'Ukraine': '18,187,531', 'Morocco': '17,946,041', 'Iraq': '17,675,043',

# The build_dataset function orchestrates the scraping process for all the additional metrics defined in the OTHER_SOURCES dictionary. It starts by calling get_country_list() to get the initial country list (which now includes 'Rank' and 'PowerIndex'). Then, it iterates through each URL and its corresponding column name in OTHER_SOURCES, scrapes the data for that metric using scrape_metric_page(), and merges this new metric into the master_df DataFrame based on the 'Country' column. A time.sleep(2) is included as a polite delay between requests to avoid overwhelming the website. Finally, it returns the complete master_df.

In [13]:
def build_dataset():
    master_df = get_country_list()

    print(f"Countries loaded: {len(master_df)}")

    for url, column_name in OTHER_SOURCES.items():
        print(f"Scraping: {column_name}")

        metric_dict = scrape_metric_page(url)

        master_df[column_name] = master_df["Country"].map(metric_dict)

        time.sleep(2)  # polite delay

    return master_df

# This main function executes the build_dataset function to gather all the data, then saves the resulting comprehensive DataFrame to a CSV file named military_raw_data.csv. This ensures all scraped data is persistently stored. The if __name__ == "__main__": block ensures that the main() function is called only when the script is executed directly, not when it's imported as a module.

In [14]:
def main():
    df = build_dataset()
    df.to_csv("military_raw_data.csv", index=False)
    print("✅ military_raw_data.csv created successfully")
if __name__ == "__main__":
    main()

Countries loaded: 145
Scraping: Total Population
Scraping: Total Military Personnel
Scraping: Active Personnel
Scraping: Reserve Personnel
Scraping: Total Aircraft Strength
Scraping: Total Helicopter Strength
Scraping: Total Tank Strength
Scraping: Total Naval Assets
Scraping: Defense Budget
✅ military_raw_data.csv created successfully


# This cell loads the military_raw_data.csv file into a Pandas DataFrame named df and then displays the first few rows (df.head()) to give a quick overview of the raw, uncleaned data.

In [15]:
df = pd.read_csv('/content/military_raw_data.csv')
print(df.columns)


Index(['Country', 'Rank', 'PowerIndex', 'Total Population',
       'Total Military Personnel', 'Active Personnel', 'Reserve Personnel',
       'Total Aircraft Strength', 'Total Helicopter Strength',
       'Total Tank Strength', 'Total Naval Assets', 'Defense Budget'],
      dtype='object')


In [16]:
df = pd.read_csv("military_raw_data.csv")
print(df.head())
df.info()

         Country  Rank  PowerIndex Total Population Total Military Personnel  \
0  United States     1      0.0741      341,963,408              150,463,900   
1         Russia     2      0.0791      140,820,810               69,002,197   
2          China     3      0.0919    1,415,043,270              764,123,366   
3          India     4      0.1346    1,409,128,296              662,290,299   
4    South Korea     5      0.1642       52,081,799               26,040,900   

  Active Personnel Reserve Personnel Total Aircraft Strength  \
0        1,333,030           799,500                  13,032   
1        1,320,000         2,000,000                   4,237   
2        2,035,000           510,000                   3,529   
3        1,431,000         1,000,000                   2,183   
4          450,000         3,100,000                   1,540   

  Total Helicopter Strength Total Tank Strength  Total Naval Assets  \
0                     5,913               4,666                

# This cell performs crucial data cleaning and type conversion. It first loads the military_raw_data.csv into a DataFrame df. Then, it identifies a list of all_potential_numeric_columns that should be numeric. It filters this list to columns_to_clean to only include columns actually present in the DataFrame. For each of these columns, it converts them to string type to handle commas and other non-numeric characters, removes commas and leading/trailing whitespace, and then converts them to numeric types using pd.to_numeric (handling errors by coercing them to NaN).

# Separately, it cleans and converts the 'Defense Budget' column by removing '$' symbols, tab characters (\t), ellipses ('...'), and commas before converting it to a numeric type. This separate treatment is due to its specific formatting. Finally, it saves the cleaned DataFrame back to military_raw_data.csv and displays its head and information to verify the changes.

In [17]:
df = pd.read_csv("military_raw_data.csv")

# List of columns to clean and convert to numeric, excluding 'Country'
all_potential_numeric_columns = [
    'Rank',
    'PowerIndex',
    'Total Population',
    'Total Military Personnel',
    'Active Personnel',
    'Reserve Personnel',
    'Total Aircraft Strength',
    'Total Helicopter Strength',
    'Total Tank Strength',
    'Total Naval Assets'
]

# Filter columns_to_clean to only include those actually present in df
columns_to_clean = [col for col in all_potential_numeric_columns if col in df.columns]

# Clean and convert columns to numeric
for col in columns_to_clean:
    df[col] = df[col].astype(str).str.replace(',', '').str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Clean and convert 'Defense Budget' separately due to '$' and '\t' characters
if 'Defense Budget' in df.columns:
    df['Defense Budget'] = df['Defense Budget'].astype(str).str.replace('$', '', regex=False)
    df['Defense Budget'] = df['Defense Budget'].str.replace('\t', '', regex=False)
    df['Defense Budget'] = df['Defense Budget'].str.replace('...', '', regex=False)
    df['Defense Budget'] = df['Defense Budget'].str.replace(',', '', regex=False).str.strip()
    df['Defense Budget'] = pd.to_numeric(df['Defense Budget'], errors='coerce')

# Save the cleaned DataFrame back to CSV
df.to_csv("military_cleaned.csv", index=False)

print("Cleaned and converted DataFrame head:")
print(df.head())
print("\nCleaned and converted DataFrame info:")
df.info()

Cleaned and converted DataFrame head:
         Country  Rank  PowerIndex  Total Population  \
0  United States     1      0.0741         341963408   
1         Russia     2      0.0791         140820810   
2          China     3      0.0919        1415043270   
3          India     4      0.1346        1409128296   
4    South Korea     5      0.1642          52081799   

   Total Military Personnel  Active Personnel  Reserve Personnel  \
0                 150463900           1333030             799500   
1                  69002197           1320000            2000000   
2                 764123366           2035000             510000   
3                 662290299           1431000            1000000   
4                  26040900            450000            3100000   

   Total Aircraft Strength  Total Helicopter Strength  Total Tank Strength  \
0                    13032                       5913                 4666   
1                     4237                       1643       

In [18]:
master_df = get_country_list()   # contains Country, Rank, PowerIndex


In [19]:
for url, column_name in OTHER_SOURCES.items():
    metric_dict = scrape_metric_page(url)
    master_df[column_name] = master_df["Country"].map(metric_dict)


In [20]:
master_df.to_csv("military_raw_data.csv", index=False)


In [21]:
df = get_country_list()
print("After country list:", df.columns)

df = build_dataset()
print("After build dataset:", df.columns)


After country list: Index(['Country', 'Rank', 'PowerIndex'], dtype='object')
Countries loaded: 145
Scraping: Total Population
Scraping: Total Military Personnel
Scraping: Active Personnel
Scraping: Reserve Personnel
Scraping: Total Aircraft Strength
Scraping: Total Helicopter Strength
Scraping: Total Tank Strength
Scraping: Total Naval Assets
Scraping: Defense Budget
After build dataset: Index(['Country', 'Rank', 'PowerIndex', 'Total Population',
       'Total Military Personnel', 'Active Personnel', 'Reserve Personnel',
       'Total Aircraft Strength', 'Total Helicopter Strength',
       'Total Tank Strength', 'Total Naval Assets', 'Defense Budget'],
      dtype='object')


In [22]:
df

,Country,Rank,PowerIndex,Total Population,Total Military Personnel,Active Personnel,Reserve Personnel,Total Aircraft Strength,Total Helicopter Strength,Total Tank Strength,Total Naval Assets,Defense Budget
0,United States,1,0.0741,"341,963,408","150,463,900","1,333,030","799,500","13,032","5,913","4,666",465,"$ \t\t\t\t\t\t\t831,500..."
1,Russia,2,0.0791,"140,820,810","69,002,197","1,320,000","2,000,000","4,237","1,643","5,630",747,"$ \t\t\t\t\t\t\t212,638..."
2,China,3,0.0919,"1,415,043,270","764,123,366","2,035,000","510,000","3,529","1,007","5,870",841,"$ \t\t\t\t\t\t\t303,000..."
3,India,4,0.1346,"1,409,128,296","662,290,299","1,431,000","1,000,000","2,183",594,"3,913",343,"$ \t\t\t\t\t\t\t109,000..."
4,South Korea,5,0.1642,"52,081,799","26,040,900","450,000","3,100,000","1,540",827,"1,831",215,"$ \t\t\t\t\t\t\t44,800,..."
...,...,...,...,...,...,...,...,...,...,...,...,...
140,Liberia,141,3.9275,"5,437,249","2,392,390","2,100",0,0,0,0,6,"$ \t\t\t\t\t\t\t20,700,000"
141,Suriname,142,4.0538,"646,758","142,287","2,500",0,3,3,0,17,"$ \t\t\t\t\t\t\t58,440,000"
142,Central African Republic,143,4.2381,"5,650,957","2,203,873","30,000",0,8,0,0,0,"$ \t\t\t\t\t\t\t300,000..."
143,Beliz,144,4.3602,"415,789","158,000","2,000",850,3,1,0,15,"$ \t\t\t\t\t\t\t34,400,000"


# This cell uses the describe() method on the DataFrame df to generate descriptive statistics for all numerical columns. This output provides a quick statistical summary, including measures like count, mean, standard deviation, and quartiles, which are essential for understanding the central tendency, dispersion, and shape of the data.

In [23]:
print(df.describe())

              Country Rank PowerIndex Total Population  \
count             145  145        145              145   
unique            145  145        145              145   
top     United States    1     0.0741      341,963,408   
freq                1    1          1                1   

       Total Military Personnel Active Personnel Reserve Personnel  \
count                       145              145               145   
unique                      145              126                85   
top                 150,463,900           25,000                 0   
freq                          1                5                52   

       Total Aircraft Strength Total Helicopter Strength Total Tank Strength  \
count                      145                       145                 145   
unique                     112                        95                 100   
top                         20                         0                   0   
freq                         4        

# This code, df.isnull().sum(), is used to check for and count missing values in your DataFrame df.

# df.isnull() generates a boolean DataFrame of the same shape as df, where True indicates a missing (NaN) value and False indicates a non-missing value.
# .sum() then counts the number of True values for each column in this boolean DataFrame. The result is a Series where the index is the column name and the value is the total count of missing entries in that column.
# This is a quick and effective way to get an overview of data completeness across your dataset.

In [24]:
df.isnull().sum()

,0
Country,0
Rank,0
PowerIndex,0
Total Population,0
Total Military Personnel,0
Active Personnel,0
Reserve Personnel,0
Total Aircraft Strength,0
Total Helicopter Strength,0
Total Tank Strength,0


In [25]:
import pandas as pd

df = pd.read_csv("/content/military_cleaned.csv")
print(df.head())
print(df.columns)


         Country  Rank  PowerIndex  Total Population  \
0  United States     1      0.0741         341963408   
1         Russia     2      0.0791         140820810   
2          China     3      0.0919        1415043270   
3          India     4      0.1346        1409128296   
4    South Korea     5      0.1642          52081799   

   Total Military Personnel  Active Personnel  Reserve Personnel  \
0                 150463900           1333030             799500   
1                  69002197           1320000            2000000   
2                 764123366           2035000             510000   
3                 662290299           1431000            1000000   
4                  26040900            450000            3100000   

   Total Aircraft Strength  Total Helicopter Strength  Total Tank Strength  \
0                    13032                       5913                 4666   
1                     4237                       1643                 5630   
2                   

In [26]:
df.dtypes


,0
Country,object
Rank,int64
PowerIndex,float64
Total Population,int64
Total Military Personnel,int64
Active Personnel,int64
Reserve Personnel,int64
Total Aircraft Strength,int64
Total Helicopter Strength,int64
Total Tank Strength,int64


# MODULE-4


In [27]:
import pandas as pd

df = pd.read_csv("military_cleaned.csv")
print(df.columns)


Index(['Country', 'Rank', 'PowerIndex', 'Total Population',
       'Total Military Personnel', 'Active Personnel', 'Reserve Personnel',
       'Total Aircraft Strength', 'Total Helicopter Strength',
       'Total Tank Strength', 'Total Naval Assets', 'Defense Budget'],
      dtype='object')


In [28]:
print(country_df.head())

         Country Rank PowerIndex
0  United States    1     0.0741
1         Russia    2     0.0791
2          China    3     0.0919
3          India    4     0.1346
4    South Korea    5     0.1642


In [29]:
import pandas as pd

df = pd.read_csv("military_cleaned.csv")
df.head()


,Country,Rank,PowerIndex,Total Population,Total Military Personnel,Active Personnel,Reserve Personnel,Total Aircraft Strength,Total Helicopter Strength,Total Tank Strength,Total Naval Assets,Defense Budget
0,United States,1,0.0741,341963408,150463900,1333030,799500,13032,5913,4666,465,831500000000
1,Russia,2,0.0791,140820810,69002197,1320000,2000000,4237,1643,5630,747,212638272000
2,China,3,0.0919,1415043270,764123366,2035000,510000,3529,1007,5870,841,303000000000
3,India,4,0.1346,1409128296,662290299,1431000,1000000,2183,594,3913,343,109000000000
4,South Korea,5,0.1642,52081799,26040900,450000,3100000,1540,827,1831,215,44800000000


In [30]:
df = pd.read_csv("military_cleaned.csv")


In [31]:
df.columns


Index(['Country', 'Rank', 'PowerIndex', 'Total Population',
       'Total Military Personnel', 'Active Personnel', 'Reserve Personnel',
       'Total Aircraft Strength', 'Total Helicopter Strength',
       'Total Tank Strength', 'Total Naval Assets', 'Defense Budget'],
      dtype='object')

Step 1: Rename Columns (for Easy Handling)

In [32]:
df = df.rename(columns={
    "Country": "country",
    "Rank": "power_rank",
    "PowerIndex": "power_index",
    "Total Population": "population",
    "Total Military Personnel": "total_military_personnel",
    "Active Personnel": "active_personnel",
    "Reserve Personnel": "reserve_personnel",
    "Total Aircraft Strength": "aircraft",
    "Total Helicopter Strength": "helicopters",
    "Total Tank Strength": "tanks",
    "Total Naval Assets": "naval_assets",
    "Defense Budget": "defense_budget"
})

Step 2: Total Military Assets

In [33]:
df["total_assets"] = df["aircraft"] + df["helicopters"] + df["tanks"] + df["naval_assets"]

Step 3: KPI 1 — Assets per Capita

In [34]:
df["assets_per_capita"] = df["total_assets"] / df["population"]

Step 4: KPI 2 — Power Index Rank Gap

In [35]:
df["power_index_rank_gap"] = df["power_rank"] - 1

 Step 5: GDP Missing → Budget Ratio Alternative

1.You do not have GDP column, so official Budget-to-GDP Ratio cannot be calculated.

2.For your evaluation, you can compute:

3.Defense Budget per Soldier (better KPI)

In [36]:
df["budget_per_soldier"] = df["defense_budget"] / df["total_military_personnel"]

OR

GDP not available from source; budget efficiency KPIs used instead.

In [37]:
df["budget_per_capita"] = df["defense_budget"] / df["population"]

Step 6: Add Continent Metadata

In [38]:
continent_map = {
    "India": "Asia", "China": "Asia", "Japan": "Asia",
    "United States": "North America", "Canada": "North America",
    "United Kingdom": "Europe", "France": "Europe", "Germany": "Europe", "Russia": "Europe",
    "Brazil": "South America",
    "Australia": "Oceania"
}

df["continent"] = df["country"].map(continent_map).fillna("Other")

Step 7: NATO Alliance Flag

In [39]:
nato_members = [
    "Albania", "Belgium", "Bulgaria", "Canada", "Croatia", "Czechia",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Italy", "Latvia", "Lithuania", "Luxembourg",
    "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland",
    "Portugal", "Romania", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Turkiye", "United Kingdom", "United States"
]

df["is_nato"] = df["country"].apply(lambda x: 1 if x in nato_members else 0)

Step 8: Tableau Long Format

In [40]:
kpi_cols = [
    "assets_per_capita",
    "power_index_rank_gap",
    "budget_per_capita"
]

df_long = df.melt(
    id_vars=["country", "continent", "is_nato"],
    value_vars=kpi_cols,
    var_name="kpi_name",
    value_name="kpi_value"
)

Step 9: Save Final Excel

In [41]:
with pd.ExcelWriter("military_final.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Wide_Data", index=False)
    df_long.to_excel(writer, sheet_name="Long_KPI_Data", index=False)

In [42]:
df = pd.read_csv("/content/military_cleaned.csv")
print(df['Country'].unique())

['United States' 'Russia' 'China' 'India' 'South Korea' 'France' 'Japan'
 'United Kingdom' 'Turkiye' 'Italy' 'Brazil' 'Germany' 'Indonesia'
 'Pakistan' 'Israel' 'Iran' 'Australia' 'Spain' 'Egypt' 'Ukraine' 'Poland'
 'Taiwan' 'Vietnam' 'Thailand' 'Saudi Arabia' 'Sweden' 'Algeria' 'Canada'
 'Singapore' 'Greece' 'North Korea' 'Argentina' 'Nigeria' 'Netherlands'
 'Myanmar' 'Mexico' 'Bangladesh' 'Portugal' 'Norway' 'South Africa'
 'Philippines' 'Malaysia' 'Colombia' 'Iraq' 'Denmark' 'Switzerland'
 'Ethiopia' 'Finland' 'Chile' 'Peru' 'Venezuela' 'Romania' 'Uzbekistan'
 'United Arab Emirates' 'Czechia' 'Morocco' 'Hungary' 'Kazakhstan'
 'Angola' 'Azerbaijan' 'Belgium' 'Bulgaria' 'Serbia'
 'Democratic Republic of the Congo' 'Cuba' 'Sudan' 'Austria' 'Sri Lanka'
 'Slovakia' 'Belarus' 'Qatar' 'Ecuador' 'Croatia' 'Jordan' 'Bahrain'
 'Kuwait' 'Albania' 'Turkmenistan' 'Tunisia' 'Libya' 'Paraguay' 'Bolivia'
 'Cambodia' 'Kenya' 'Chad' 'Oman' 'Syria' 'Lithuania' 'Tanzania'
 'New Zealand' 'Slovenia' 'Moz